# 06 · Operation count of one LoRA training step (Table 2)

We verify the analytic counts of Section 2.4 with PyTorch's `FlopCounterMode`, which counts every matrix
product executed in the forward and the backward pass (it reports $2nmk$ floating-point operations for an
$n\times m$ by $m\times k$ product; we divide by 2 to use the coursework's $nmk$ convention).
Three ways of training one layer $W\in\mathbb{R}^{d_{out}\times d_{in}}$ on a batch of $N$ inputs:
full fine-tuning; LoRA evaluated as $B(AX)$; LoRA evaluated as $(BA)X$.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import torch
import torch.nn as nn
from torch.utils.flop_counter import FlopCounterMode

from krypto.lora import LoRALinear, ops_full, ops_lora_BAx, ops_lora_BA_x, load_base, add_lora, full_finetune
from krypto.stats import save_json
from krypto.energy import track

torch.manual_seed(0)


class LoRAMerged(nn.Module):
    """LoRA evaluated left to right: forms M = B A (d_out x d_in) and then (W + M) X."""

    def __init__(self, base: nn.Linear, r: int):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.randn(base.out_features, r) * 0.01)

    def forward(self, x):
        return nn.functional.linear(x, self.base.weight + self.B @ self.A, self.base.bias)


def count(module, d_in, N, input_grad):
    x = torch.randn(N, d_in, requires_grad=input_grad)
    with FlopCounterMode(display=False) as fwd:
        out = module(x)
    with FlopCounterMode(display=False) as bwd:
        out.sum().backward()
    return fwd.get_total_flops() // 2, bwd.get_total_flops() // 2


def check(d_in, d_out, N, r, input_grad):
    rows = []
    for name, module, formula in (
            ("full fine-tuning", nn.Linear(d_in, d_out), ops_full(d_in, d_out, N, input_grad)),
            ("LoRA, B(AX)", LoRALinear(nn.Linear(d_in, d_out), r), ops_lora_BAx(d_in, d_out, N, r, input_grad)),
            ("LoRA, (BA)X", LoRAMerged(nn.Linear(d_in, d_out), r), ops_lora_BA_x(d_in, d_out, N, r, input_grad))):
        f, b = count(module, d_in, N, input_grad)
        assert (f, b) == (formula["forward"], formula["backward"]), (name, f, b, formula)
        rows.append((name, f, b, f + b, formula["params"]))
    return rows

## Measured counts agree exactly with the formulas

In [2]:
table = {}
with track("06_flop_counts"):
    for (d_in, d_out, input_grad, layer) in ((512, 512, True, "hidden layer 512x512"),
                                             (40, 512, False, "first layer 512x40")):
        for N, r in ((64, 1), (64, 4), (64, 64), (1024, 4)):
            rows = check(d_in, d_out, N, r, input_grad)
            table[f"{layer}|N={N}|r={r}"] = rows
            full_total = rows[0][3]
            print(f"\n{layer}, N={N}, r={r}  (input gradient {'needed' if input_grad else 'not needed'})")
            for name, f, b, t, p in rows:
                print(f"   {name:17s} forward {f:>12,}  backward {b:>12,}  total {t:>12,}  "
                      f"({t / full_total:5.3f} x full)   trainable params {p:,}")
print("\nAll measured counts match the analytic formulas.")
save_json({k: [list(r) for r in v] for k, v in table.items()}, "lora_flops.json")


hidden layer 512x512, N=64, r=1  (input gradient needed)
   full fine-tuning  forward   16,777,216  backward   33,554,432  total   50,331,648  (1.000 x full)   trainable params 262,656
   LoRA, B(AX)       forward   16,842,752  backward   16,908,288  total   33,751,040  (0.671 x full)   trainable params 1,024
   LoRA, (BA)X       forward   17,039,360  backward   34,078,720  total   51,118,080  (1.016 x full)   trainable params 1,024

hidden layer 512x512, N=64, r=4  (input gradient needed)
   full fine-tuning  forward   16,777,216  backward   33,554,432  total   50,331,648  (1.000 x full)   trainable params 262,656
   LoRA, B(AX)       forward   17,039,360  backward   17,301,504  total   34,340,864  (0.682 x full)   trainable params 4,096
   LoRA, (BA)X       forward   17,825,792  backward   35,651,584  total   53,477,376  (1.062 x full)   trainable params 4,096

hidden layer 512x512, N=64, r=64  (input gradient needed)
   full fine-tuning  forward   16,777,216  backward   33,554,432 

## Whole-network cost of one training step of KryptoNet-40 (batch $N=64$)
For LoRA on $W_1$ only, the backward pass must still propagate gradients through the four frozen layers above it
(input gradients only, no weight gradients).

In [3]:
base = load_base()
configs = {"full fine-tuning": full_finetune(base), "LoRA W1, r=1": add_lora(base, 1, (0,)),
           "LoRA W1-W4, r=4": add_lora(base, 4, (0, 2, 4, 6))}
net_rows = {}
x = torch.randn(64, 40)
yb = torch.randint(0, 2, (64,)).float()
for name, net in configs.items():
    with FlopCounterMode(display=False) as fwd:
        out = net(x).squeeze(1)
    loss = nn.functional.binary_cross_entropy_with_logits(out, yb)
    with FlopCounterMode(display=False) as bwd:
        loss.backward()
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=1e-3)
    times = []
    for _ in range(300):
        t0 = time.perf_counter()
        opt.zero_grad(); nn.functional.binary_cross_entropy_with_logits(net(x).squeeze(1), yb).backward(); opt.step()
        times.append(time.perf_counter() - t0)
    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    net_rows[name] = (fwd.get_total_flops() // 2, bwd.get_total_flops() // 2, trainable, float(np.median(times[50:])))
for name, (f, b, p, t) in net_rows.items():
    full = sum(net_rows["full fine-tuning"][:2])
    print(f"{name:17s} forward {f:>11,}  backward {b:>11,}  total {f + b:>11,} ({(f + b) / full:.3f} x)  "
          f"trainable {p:>8,}  Adam state {2 * p:>9,}  median step {1e3 * t:.2f} ms")
save_json(net_rows, "lora_network_cost.json")

full fine-tuning  forward  51,675,136  backward 102,039,552  total 153,714,688 (1.000 x)  trainable  809,473  Adam state 1,618,946  median step 5.70 ms
LoRA W1, r=1      forward  51,710,464  backward  50,432,512  total 102,142,976 (0.664 x)  trainable      552  Adam state     1,104  median step 2.16 ms
LoRA W1-W4, r=4   forward  52,602,880  backward  52,209,664  total 104,812,544 (0.682 x)  trainable   14,496  Adam state    28,992  median step 2.93 ms
